# MRI preprocessing walkthrough

Run top to bottom to inspect each stage of the MRI preprocessing pipeline:

1. Read the source scan and reorient it to RPS.
2. N4 bias correction using a Li foreground mask.
3. DeepBrain extraction (`probability > 0.3`).
4. Align brain-mask centroids, then run deterministic rigid and affine registration to the MNI152 template. Reject fits that crop more than 1% of the extracted brain.
5. `SimpleITK.Normalize` over the complete registered image.
6. Linear resize to `128 × 128 × 128`, preserving physical extent.

The brain mask guides alignment and is also saved for inspection. Native-space and template-space panels use different grids.

In [1]:
from pathlib import Path
import importlib
import sys
import matplotlib.pyplot as plt
import numpy as np
import nibabel as nib

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'preprocessing').is_dir():
    REPO_ROOT = Path.cwd().parent
# Put this checkout first, then evict cached preprocessing modules so the
# kernel cannot reuse the earlier HD-BET/2 mm implementation.
sys.path.insert(0, str(REPO_ROOT.resolve()))
for module_name in tuple(sys.modules):
    if module_name == 'preprocessing' or module_name.startswith('preprocessing.'):
        del sys.modules[module_name]
importlib.invalidate_caches()

import preprocessing.config as preprocessing_config
import preprocessing.pipeline as preprocessing_pipeline
from preprocessing.config import PipelineConfig
from preprocessing.pipeline import preprocess_mri_scan
print('Loaded pipeline:', preprocessing_pipeline.__file__)
print('Pipeline target grid:', PipelineConfig().target_shape)

INPUT_SCAN = REPO_ROOT / 'data/raw_scans/141_S_4426/MPRAGE.nii.gz'  # Change to another raw T1 scan
TEMPLATE = REPO_ROOT / 'templates/mni152.nii.gz'
OUTPUT_DIR = REPO_ROOT / 'data/preprocessed_walkthrough'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / f'{INPUT_SCAN.name.removesuffix(".nii.gz").removesuffix(".nii")}_128.nii.gz'

assert INPUT_SCAN.is_file(), f'Missing input scan: {INPUT_SCAN}'
assert TEMPLATE.is_file(), f'Missing MNI template: {TEMPLATE}'
print('Input:', INPUT_SCAN)
print('MNI template:', TEMPLATE)


Loaded pipeline: E:\Desktop\git_repos\DG-TCAV\preprocessing\pipeline.py
Pipeline target grid: (128, 128, 128)
Input: e:\Desktop\git_repos\DG-TCAV\data\raw_scans\141_S_4426\MPRAGE.nii.gz
MNI template: e:\Desktop\git_repos\DG-TCAV\templates\mni152.nii.gz


## Run preprocessing and capture its stages

In [2]:
config = PipelineConfig(template_path=TEMPLATE, target_shape=(128, 128, 128))
if not (preprocess_mri_scan.__doc__ or '').startswith('Run orientation, bias correction, extraction'):
    raise RuntimeError(
        'This kernel still has an older preprocessing module imported. Restart the kernel, '
        'reopen this updated notebook, and run its setup cell first.'
    )
assert config.target_shape == (128, 128, 128)
assert config.template_path.resolve() == TEMPLATE.resolve()
result = preprocess_mri_scan(
    str(INPUT_SCAN),
    output_path=str(OUTPUT_PATH),
    collect_steps=True,
    config=config,
) 
steps = result.step_volumes
print('Pipeline completed successfully.')
print('Output:', OUTPUT_PATH)
print('Shape:', result.final_shape)
print('Spacing (mm):', tuple(round(v, 4) for v in result.voxel_spacing))
print('Elapsed (s):', round(result.elapsed_sec, 1))



Instructions for updating:
non-resource variables are not supported in the long term
Instructions for updating:
Use tf.gfile.GFile.
Pipeline completed successfully.
Output: e:\Desktop\git_repos\DG-TCAV\data\preprocessed_walkthrough\MPRAGE_128.nii.gz
Shape: (128, 128, 128)
Spacing (mm): (1.1926, 1.4749, 1.2387)
Elapsed (s): 99.7


## Visualize each processing stage

In [ ]:
def show_stage(key, title, mask_key=None):
    volume = np.asarray(steps[key], dtype=np.float32)
    sx, sy, sz = result.step_spacings[key]
    i, j, k = (n // 2 for n in volume.shape)
    planes = [volume[:, :, k].T, volume[:, j, :].T, volume[i, :, :].T]
    # imshow's aspect is vertical voxel size / horizontal voxel size.
    # This displays each plane in millimeters, despite anisotropic voxels.
    aspects = [sy / sx, sz / sx, sz / sy]
    labels = ['Axial', 'Coronal', 'Sagittal']
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, plane, label, aspect in zip(axes, planes, labels, aspects):
        ax.imshow(plane, cmap='gray', origin='lower', aspect=aspect)
        ax.set_title(label)
        ax.axis('off')
    if mask_key and mask_key in steps:
        mask = np.asarray(steps[mask_key], dtype=bool)
        mask_planes = [mask[:, :, k].T, mask[:, j, :].T, mask[i, :, :].T]
        for ax, mask_plane in zip(axes, mask_planes):
            ax.contour(mask_plane, levels=[0.5], colors=['cyan'], linewidths=0.7)
    fig.suptitle(f'{title} — array shape {volume.shape}; spacing (mm) {result.step_spacings[key]}')
    plt.tight_layout()
    plt.show()

show_stage('0_raw', '0. Raw input')
show_stage('1_rps', '1. RPS orientation')
show_stage('2_n4', '2. N4 bias-field correction', '2_head_mask')
show_stage('3_stripped', '3. DeepBrain skull stripping', '3_mask')
show_stage('4_registered', '4. Masked rigid + affine registration to the MNI152 template', '4_mask')
show_stage('5_normalized', '5. SimpleITK.Normalize')
show_stage('6_resized', '6. Resize to 128³', '6_mask')


## Inspect saved output geometry and mask

In [ ]:
out = nib.load(str(OUTPUT_PATH))
mask_path = OUTPUT_PATH.with_name(OUTPUT_PATH.name.removesuffix('.nii.gz') + '_mask.nii.gz')
mask_img = nib.load(str(mask_path))
print('Image shape:', out.shape)
print('Image spacing (mm):', out.header.get_zooms()[:3])
print('Mask shape:', mask_img.shape)
print('Mask file:', mask_path)
print('MNI template shape:', nib.load(str(TEMPLATE)).shape)


Image shape: (128, 128, 128)
Image spacing (mm): (np.float32(1.1926161), np.float32(1.4749262), np.float32(1.2387075))
Mask shape: (128, 128, 128)
Mask file: e:\Desktop\git_repos\DG-TCAV\data\preprocessed_walkthrough\MPRAGE_128_mask.nii.gz
MNI template shape: (207, 256, 215)
